# Collect financial news headlines
Runs `scripts/03_collect_financial_news.py` end-to-end on Google Colab.


## 1. Mount Drive & clone repo
Set `USE_DRIVE=True` to persist `data/`, `results/`, `figures/` across Colab runtime resets. 
Set `REPO_URL` to override the auto-detected git remote.


In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/CARA-FinSent"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}  # leave blank to auto-detect from this notebook's repo

import os, subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    WORK_DIR = Path(DRIVE_DIR)
else:
    WORK_DIR = Path("/content")

REPO_DIR = WORK_DIR / "cara-finsent-experiments"
if not REPO_DIR.exists():
    if not REPO_URL:
        # Best-effort auto-detect: try the repo this notebook lives in (works when notebook is opened from GitHub).
        REPO_URL = os.environ.get("REPO_URL", "")
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL above (e.g. https://github.com/<user>/cara-finsent-experiments.git)")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working in:", os.getcwd())


## 2. Install dependencies


In [ ]:
!pip -q install -r requirements.txt


## 3b. Configure parameters


In [ ]:
WEAK_LABEL = True  #@param {type:"boolean"}
INCLUDE_NEWSAPI = False  #@param {type:"boolean"}
INCLUDE_FINNHUB = False  #@param {type:"boolean"}
import os, getpass
if INCLUDE_NEWSAPI and not os.environ.get("NEWSAPI_KEY"):
    os.environ["NEWSAPI_KEY"] = getpass.getpass("NEWSAPI_KEY: ")
if INCLUDE_FINNHUB and not os.environ.get("FINNHUB_API_KEY"):
    os.environ["FINNHUB_API_KEY"] = getpass.getpass("FINNHUB_API_KEY: ")
argv = []
if WEAK_LABEL: argv.append("--weak_label")
if INCLUDE_NEWSAPI: argv.append("--include_newsapi")
if INCLUDE_FINNHUB: argv.append("--include_finnhub")


## 3c. Run `03_collect_financial_news.py`


In [ ]:
import sys, runpy
sys.argv = ['scripts/03_collect_financial_news.py'] + argv
print("Running:", " ".join(sys.argv))
runpy.run_path("scripts/03_collect_financial_news.py", run_name="__main__")


## ⤓ Download results
Zip `results/` + `figures/` for sharing.


In [ ]:
import shutil, datetime
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
archive = shutil.make_archive(f"cara_results_{ts}", "zip", root_dir=".", base_dir="results")
print("Created:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
